# Упражнение: остановить станок или нет

**Дисциплина «ИИ и машинное обучение»** · НИУ ВШЭ, Высшая школа бизнеса

Магистратура «Бизнес-аналитика и цифровые решения в финансах»

---

Фрезерный станок в цеху обвешан датчиками: температура, обороты
шпинделя, крутящий момент, износ резца. Мастер смены решает, когда
остановить станок на профилактику. Остановить рано — линия простаивает
зря. Опоздать — станок встанет сам, посреди партии, и простой будет
в разы дольше.

Здесь вы строите модель, которая по показаниям датчиков оценивает
вероятность отказа, и превращаете эту вероятность в решение
«останавливать или нет». Качество модели, и цена ошибки меряются **часами простоя линии**.

| Часть | О чём |
|---|---|
| 1 | Задача, данные и признаки |
| 2 | Лестница моделей: физика станка превращается в признаки |
| 3 | Порог: сколько часов простоя стоит ошибка в каждую сторону |

**Как работать.** Ячейки с заголовком **Задание** содержат `...` или
`raise NotImplementedError` — это место надо заменить своим кодом.
Самопроверка стоит в конце той же ячейки: если всё верно, она печатает
`✓ верно`. Ячейки выполняются сверху вниз: каждая опирается на результат
предыдущей.

In [1]:
## ── библиотеки ─────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# конвейер: последовательность «подготовка данных → модель»
from sklearn.pipeline import Pipeline
# своя обработка для каждой группы столбцов
from sklearn.compose import ColumnTransformer
# масштабирование чисел, кодирование категорий, попарные произведения
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
# модель для ответа «да или нет»
from sklearn.linear_model import LogisticRegression
# деление выборки на части
from sklearn.model_selection import train_test_split
# метрики для классификации
from sklearn.metrics import (roc_auc_score, accuracy_score,
                             confusion_matrix, roc_curve)

## ── настройки печати и графиков ────────────────────────────────────
np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.width", 120)
plt.rcParams.update({
    "figure.figsize": (7.2, 3.8), "figure.dpi": 110, "font.size": 11,
    "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False,
})
BLUE, INK, RED, GREY, GREEN = "#0057b8", "#0d2340", "#c0392b", "#8a9bb0", "#1b7f5a"
SEED = 42

## ── вспомогательные функции ────────────────────────────────────────
LADDER = []


def check(ok, hint):
    "Самопроверка задания."
    if not ok:
        raise AssertionError("✗ " + hint)
    print("✓ верно")


def score(tag, y_true, proba, remember=True):
    "Печатает качество прогноза и запоминает шаг для итоговой лестницы."
    auc = roc_auc_score(y_true, proba)
    acc = accuracy_score(y_true, proba >= 0.5)
    if remember:
        LADDER.append((tag, auc))
    print("%s: AUC %.3f, accuracy при пороге 0,5 %.4f" % (tag, auc, acc))
    return auc


print("Библиотеки загружены.")

c:\Users\temur\miniconda3\envs\optinet\lib\site-packages\sklearn\utils\_param_validation.py:14: UserWarning: A NumPy version >=1.23.5 and <2.5.0 is required for this version of SciPy (detected version 1.23.2)
  from scipy.sparse import csr_matrix, issparse


Библиотеки загружены.


---
# Часть 1. Задача и данные

## Шаг 0. Что мы решаем

Профилактика по регламенту — это остановка на два часа: резец меняют,
станок чистят, линия стоит. Аварийный отказ посреди партии — это
шестнадцать часов: пока бригада доедет, пока разберут, пока доделают
испорченную деталь.

Обе ошибки стоят простоя, и обе плохи по-разному:

* **остановили зря** — потеряли два часа на станке, который работал бы
  дальше;
* **не остановили вовремя** — потеряли шестнадцать.

Значит, перед моделью стоит не вопрос «сломается или нет», а вопрос
**«стоит ли останавливать»** — и ответ на него зависит от того, во что
обходится каждая из ошибок. К этому мы придём в третьей части.

Мерить качество самой модели будем **ROC-AUC**. Почему не долей верных
ответов, увидим прямо сейчас.

Данные — 10 000 наблюдений за фрезерным станком: показания датчиков
в момент изготовления детали и отметка, случился ли отказ. Набор
открытый (AI4I 2020 Predictive Maintenance, UCI,
[archive.ics.uci.edu/dataset/601](https://archive.ics.uci.edu/dataset/601/ai4i+2020+predictive+maintenance+dataset)),
в репозитории курса лежит его готовая версия: названия переведены на
русский, температуры переведены из кельвинов в градусы Цельсия.
**Больше ничего не делали.**

In [2]:
## ── данные ─────────────────────────────────────────────────────────
# read_csv читает таблицу прямо по ссылке: скачивать ничего не нужно
MACHINE = ("https://raw.githubusercontent.com/Rimest/ml_ai_course_gsb_hse"
           "/main/data/machine_failures.csv")
станок = pd.read_csv(MACHINE)

print("Строк: %d, столбцов: %d" % станок.shape)

станок.head()

Строк: 10000, столбцов: 14


,номер,артикул,класс_детали,температура_цеха,температура_станка,обороты,момент,износ_резца,отказ,износ_резца_причина,перегрев_причина,мощность_причина,перегрузка_причина,случайный_отказ
0,1,M14860,средний,24.95,35.45,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,низкий,25.05,35.55,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,низкий,24.95,35.35,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,низкий,25.05,35.45,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,низкий,25.05,35.55,1408,40.0,9,0,0,0,0,0,0


Одна строка — одна деталь и показания станка в момент её изготовления.
Столбцы:

| Столбец | Что в нём |
|---|---|
| `номер`, `артикул` | порядковый номер строки и код детали |
| `класс_детали` | требования к точности: низкий, средний, высокий |
| `температура_цеха`, `температура_станка` | °C |
| `обороты` | обороты шпинделя в минуту |
| `момент` | крутящий момент, Н·м |
| `износ_резца` | сколько минут отработал резец |
| `отказ` | **это ответ**: 1 — станок отказал |
| `…_причина`, `случайный_отказ` | пять столбцов о том, что именно сломалось |

### Задание 1.1
Посчитайте долю отказов и долю верных ответов у того, кто вообще не
строит моделей и всегда отвечает «поломки не будет».

In [ ]:
## ── осмотр ─────────────────────────────────────────────────────────
доля_отказов = ...  # TODO: средняя доля единиц в столбце «отказ»
всегда_ноль = np.zeros(len(станок))    # прогноз «поломки не будет» для всех строк
accuracy_нуля = ...  # TODO: accuracy_score от ответа и всегда_ноль

## ── числа ──────────────────────────────────────────────────────────
print("Отказов: %d из %d (%.2f %%)"
      % (станок["отказ"].sum(), len(станок), 100 * доля_отказов))
print("Доля верных ответов у «поломки не будет»: %.4f" % accuracy_нуля)
print("")
print("Доля отказов по классу детали:")
print((100 * станок.groupby("класс_детали")["отказ"].mean()).round(2).to_string())

## ── самопроверка ───────────────────────────────────────────────────
check(abs(доля_отказов - 0.0339) < 1e-4, "доля отказов — это среднее по столбцу «отказ»")
check(abs(accuracy_нуля - 0.966) < 1e-3, "должно получиться 0,9661")

96,6 % верных ответов — и ни одной пойманной поломки. Доля верных
ответов при редком событии бесполезна: её даёт пустой прогноз.

Поэтому качество модели меряют **ROC-AUC**. Это вероятность того, что
случайно взятая строка с отказом получит от модели оценку выше, чем
случайно взятая строка без отказа. Иначе говоря, AUC отвечает на
вопрос «умеет ли модель расставлять наблюдения в правильном порядке»,
и её не обманешь пустым прогнозом: у «поломки не будет» AUC ровно 0,5,
как у подбрасывания монетки.

Порядок — это ровно то, что нужно мастеру смены: он всё равно не может
остановить станок на каждой детали, ему важно знать, какие наблюдения
тревожнее прочих.

### Задание 1.2
Разделите данные на три части: обучение 60 %, валидация 20 %, тест 20 %.
Оба раза передавайте `stratify` по столбцу `отказ` — чтобы доля отказов
в частях осталась одинаковой, — и `random_state=SEED`.

In [ ]:
## ── делим выборку ──────────────────────────────────────────────────
# TODO: два вызова train_test_split: сначала 40 % в остаток, потом остаток пополам
raise NotImplementedError("замените эту строку своим решением")
train, valid, test = train.copy(), valid.copy(), test.copy()

## ── числа ──────────────────────────────────────────────────────────
print("Части:")
for name, part in (("обучение", train), ("валидация", valid), ("тест", test)):
    print("   %-11s %5d строк, отказов %3d (%.2f %%)"
          % (name, len(part), part["отказ"].sum(), 100 * part["отказ"].mean()))

## ── самопроверка ───────────────────────────────────────────────────
check(len(train) == 6000 and len(valid) == 2000 and len(test) == 2000,
      "должно получиться 6 000 / 2 000 / 2 000 строк")
check(train["отказ"].sum() == 203, "в обучении должно оказаться 203 отказа")

Доля отказов во всех трёх частях одинаковая — этого мы и добивались
через `stratify`. Без него в проверочной части могло бы случайно
оказаться вдвое меньше отказов, и все измерения поехали бы.

## Шаг 1. Сначала попробуем всё подряд

Прежде чем выбирать признаки, поставим опыт: отдадим модели все
числовые столбцы, какие есть в таблице, и посмотрим, что получится.

`LogisticRegression(max_iter=3000)` — линейная модель для ответа
«да или нет»: она считает взвешенную сумму признаков и превращает её
в вероятность от 0 до 1. `max_iter` — сколько шагов даём на подбор
весов; со значением по умолчанию модель не всегда успевает сойтись.
Метод `predict_proba(X)[:, 1]` возвращает вероятность отказа.

### Задание 1.3
Обучите логистическую регрессию на всех числовых столбцах из списка
`ВСЕ_ЧИСЛА` и замерьте качество на валидации.

In [ ]:
## ── все числовые столбцы, какие есть ───────────────────────────────
ВСЕ_ЧИСЛА = ["номер", "температура_цеха", "температура_станка", "обороты",
             "момент", "износ_резца", "износ_резца_причина",
             "перегрев_причина", "мощность_причина", "перегрузка_причина",
             "случайный_отказ"]

model = Pipeline([
    ("масштаб", StandardScaler()),
    ("модель", LogisticRegression(max_iter=3000)),
])
# TODO: обучите model на train[ВСЕ_ЧИСЛА] и ответе train["отказ"], затем возьмите вероятности на валидации
raise NotImplementedError("замените эту строку своим решением")

## ── качество на валидации ──────────────────────────────────────────
score("всё подряд", valid["отказ"], proba, remember=False)

## ── на что модель опирается ────────────────────────────────────────
веса = pd.Series(model.named_steps["модель"].coef_[0], index=ВСЕ_ЧИСЛА)
print("")
print("Самые влиятельные признаки:")
for имя, вес in веса.sort_values(key=abs, ascending=False).head(5).items():
    print("   %-22s %+.2f" % (имя, вес))

## ── самопроверка ───────────────────────────────────────────────────
check(roc_auc_score(valid["отказ"], proba) > 0.95, "AUC должен получиться около 0,974")

Почти идеально: 99,85 % верных ответов и AUC 0,974. И на этом месте
работу надо остановить и посмотреть, на что модель опирается.

Первые четыре места заняли столбцы с причинами отказа. Посмотрите, как
они устроены: `перегрев_причина` равен единице ровно тогда, когда
станок отказал из-за перегрева. Эти столбцы заполняет ремонтная
бригада **после** поломки, разобрав станок. В тот момент, когда мастер
решает, останавливать ли линию, их не существует.

Это утечка: ответ, переписанный другими словами. Модель, обученная на
таких признаках, в цеху бесполезна — там ей будет нечем их заполнить.

Заодно уберём два столбца, которые ничего не значат: `номер` — это
порядковый номер строки в файле, `артикул` — код детали. Порядковый
номер иногда выглядит полезным (в данных, собранных по времени, он
заменяет дату), и как раз поэтому его стоит выкидывать осознанно,
а не оставлять «на всякий случай».

Остаётся то, что действительно видно в цеху до поломки: показания
датчиков и класс детали.

---
# Часть 2. Лестница моделей

Дальше всё устроено одинаково: строим модель, меряем AUC **на
валидации**, записываем результат и думаем, чего модели не хватает.

### Задание 2.1
Обучите модель на пяти показаниях датчиков.

In [ ]:
## ── признаки ───────────────────────────────────────────────────────
ФИЗИКА = ["температура_цеха", "температура_станка", "обороты", "момент",
          "износ_резца"]

model = Pipeline([
    ("масштаб", StandardScaler()),
    ("модель", LogisticRegression(max_iter=3000)),
])
# TODO: обучите на train[ФИЗИКА] и посчитайте вероятности отказа на валидации
raise NotImplementedError("замените эту строку своим решением")

## ── качество на валидации ──────────────────────────────────────────
auc = score("1. пять датчиков", valid["отказ"], proba)

## ── самопроверка ───────────────────────────────────────────────────
check(auc > 0.85, "AUC должен получиться около 0,875")

0,875 — модель уверенно расставляет наблюдения по порядку, хотя
доля верных ответов у неё почти такая же, как у пустого прогноза
(0,971 против 0,966). Это нормально: при пороге 0,5 она почти никого
не останавливает, а порогом мы займёмся в третьей части.

Посмотрим на ROC-кривую — картинку, из которой берётся AUC.

In [ ]:
## ── ROC-кривая ─────────────────────────────────────────────────────
доля_ложных, доля_пойманных, _ = roc_curve(valid["отказ"], proba)

fig, ax = plt.subplots(figsize=(5.4, 4.6))
ax.plot(доля_ложных, доля_пойманных, color=BLUE, lw=2, label="модель, AUC %.3f" % auc)
ax.plot([0, 1], [0, 1], color=GREY, ls="--", label="монетка, AUC 0,500")
ax.set_xlabel("доля зря остановленных станков")
ax.set_ylabel("доля пойманных отказов")
ax.set_title("ROC-кривая на валидации")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

Каждая точка кривой — это какой-то порог. Двигаясь по ней вправо, мы
ловим больше отказов, но останавливаем больше исправных станков. AUC —
площадь под этой кривой, то есть качество модели сразу при всех
порогах. Какую именно точку выбрать, решает не модель, а цех.

## Шаг 2. Класс детали

Отказы распределены по классам неравномерно: на деталях низкого класса
3,9 %, высокого — 2,1 %. Добавим класс категорией.

### Задание 2.2
Соберите `ColumnTransformer`: `ФИЗИКА` через `StandardScaler`,
`класс_детали` через `OneHotEncoder`.

In [ ]:
## ── подготовка столбцов ────────────────────────────────────────────
КАТЕГОРИИ = ["класс_детали"]

# TODO: ColumnTransformer из двух частей: числа через StandardScaler, категории через OneHotEncoder
raise NotImplementedError("замените эту строку своим решением")

## ── обучение ───────────────────────────────────────────────────────
model = Pipeline([("подготовка", подготовка),
                  ("модель", LogisticRegression(max_iter=3000))])
model.fit(train, train["отказ"])

## ── качество на валидации ──────────────────────────────────────────
proba = model.predict_proba(valid)[:, 1]
auc = score("2. + класс детали", valid["отказ"], proba)

## ── самопроверка ───────────────────────────────────────────────────
check(0.86 < auc < 0.89, "AUC почти не изменится — так и должно быть")

0,874 — ровно то же, что было. Признак, у которого доля отказов заметно
различается, ничего не дал.

Объяснение в том, как класс детали влияет на поломку. Он не повышает
риск сам по себе — он задаёт **предел**, после которого перегрузка
становится опасной: у детали высокого класса запас больше. То есть
класс важен не сам, а в сочетании с нагрузкой, а линейная модель
признаки складывает, а не перемножает. Пока мы не дадим ей сочетание
в явном виде, толку от класса не будет.

## Шаг 3. Физика: мощность

Инженер, глядя на таблицу, сразу назвал бы величину, которой в ней нет.
Станок рвёт не момент сам по себе и не обороты сами по себе, а
**мощность** — произведение момента на угловую скорость:

$$N = M \cdot \omega = M \cdot \frac{2\pi \cdot \text{обороты}}{60}$$

Для линейной модели это принципиально новая информация: складывать
момент и обороты она умеет, перемножать — нет.

### Задание 2.3
Добавьте в каждую часть столбец `мощность` по формуле выше и включите
его в числовые признаки.

In [ ]:
## ── подготовка столбцов ────────────────────────────────────────────
for part in (train, valid, test):
    # TODO: момент, умноженный на обороты и на 2π/60
    raise NotImplementedError("замените эту строку своим решением")

ЧИСЛА = ФИЗИКА + ["мощность"]

подготовка = ColumnTransformer([
    ("числа", StandardScaler(), ЧИСЛА),
    ("категории", OneHotEncoder(handle_unknown="ignore"), КАТЕГОРИИ),
])

## ── обучение ───────────────────────────────────────────────────────
model = Pipeline([("подготовка", подготовка),
                  ("модель", LogisticRegression(max_iter=3000))])
model.fit(train, train["отказ"])

## ── качество на валидации ──────────────────────────────────────────
proba = model.predict_proba(valid)[:, 1]
auc = score("3. + мощность", valid["отказ"], proba)

## ── числа ──────────────────────────────────────────────────────────
print("Мощность, Вт: при нормальной работе %.0f, при отказе %.0f"
      % (train.loc[train["отказ"] == 0, "мощность"].mean(),
         train.loc[train["отказ"] == 1, "мощность"].mean()))

## ── самопроверка ───────────────────────────────────────────────────
check(auc > 0.89, "AUC должен подрасти примерно до 0,898")

0,898 — больше двух пунктов из столбца, который целиком
посчитан из двух уже имеющихся. Никаких новых измерений не появилось,
появилась новая **форма** старых.

## Шаг 4. А теперь разность

Раз произведение помогло, попробуем другую очевидную величину:
перепад температур между станком и цехом. Станок, который греется
сильнее обычного, — кандидат на перегрев.

### Задание 2.4
Добавьте столбец `перепад` — температура станка минус температура цеха
— и включите его в числовые признаки.

In [ ]:
## ── подготовка столбцов ────────────────────────────────────────────
for part in (train, valid, test):
    # TODO: температура станка минус температура цеха
    raise NotImplementedError("замените эту строку своим решением")

ЧИСЛА = ФИЗИКА + ["мощность", "перепад"]

подготовка = ColumnTransformer([
    ("числа", StandardScaler(), ЧИСЛА),
    ("категории", OneHotEncoder(handle_unknown="ignore"), КАТЕГОРИИ),
])

## ── обучение ───────────────────────────────────────────────────────
model = Pipeline([("подготовка", подготовка),
                  ("модель", LogisticRegression(max_iter=3000))])
model.fit(train, train["отказ"])

## ── качество на валидации ──────────────────────────────────────────
proba = model.predict_proba(valid)[:, 1]
auc = score("4. + перепад температур", valid["отказ"], proba)

## ── числа ──────────────────────────────────────────────────────────
print("Перепад, °C: при нормальной работе %.1f, при отказе %.1f"
      % (train.loc[train["отказ"] == 0, "перепад"].mean(),
         train.loc[train["отказ"] == 1, "перепад"].mean()))

## ── самопроверка ───────────────────────────────────────────────────
check(0.89 < auc < 0.91, "AUC останется примерно таким же — так и должно быть")

0,898 — ни одного пункта, хотя при отказе перепад заметно меньше
(9,4 против 10,0).

Причина простая, и её стоит запомнить на все линейные модели: **перепад
— это разность двух признаков, которые у модели уже есть**. Модель
считает взвешенную сумму, и разность она умеет строить сама, подобрав
весам противоположные знаки. Новой информации в таком столбце ноль.

Отсюда общее правило:

* **сумма и разность** признаков линейной модели не нужны — она их
  умеет;
* **произведение, отношение, квадрат, логарифм** — нужны, потому что
  сама она их построить не может.

Перепад мы всё же оставим: дальше он пойдёт в произведения, а там
разность температур вместе с оборотами уже кое-что значит.

## Шаг 5. Нагрузка на резец

Ещё одна величина из цеха: изношенный резец под большим моментом рвётся
скорее, чем свежий. Произведение износа на момент — мера накопленной
нагрузки.

### Задание 2.5
Добавьте столбец `нагрузка` — износ резца, умноженный на момент.

In [ ]:
## ── подготовка столбцов ────────────────────────────────────────────
for part in (train, valid, test):
    # TODO: износ резца, умноженный на момент
    raise NotImplementedError("замените эту строку своим решением")

ЧИСЛА = ФИЗИКА + ["мощность", "перепад", "нагрузка"]

подготовка = ColumnTransformer([
    ("числа", StandardScaler(), ЧИСЛА),
    ("категории", OneHotEncoder(handle_unknown="ignore"), КАТЕГОРИИ),
])

## ── обучение ───────────────────────────────────────────────────────
model = Pipeline([("подготовка", подготовка),
                  ("модель", LogisticRegression(max_iter=3000))])
model.fit(train, train["отказ"])

## ── качество на валидации ──────────────────────────────────────────
proba = model.predict_proba(valid)[:, 1]
auc = score("5. + нагрузка на резец", valid["отказ"], proba)

## ── числа ──────────────────────────────────────────────────────────
print("Нагрузка: при нормальной работе %.0f, при отказе %.0f"
      % (train.loc[train["отказ"] == 0, "нагрузка"].mean(),
         train.loc[train["отказ"] == 1, "нагрузка"].mean()))

## ── самопроверка ───────────────────────────────────────────────────
check(0.88 < auc < 0.91, "AUC почти не сдвинется")

0,895 — и снова ничего, хотя при отказе нагрузка в полтора с лишним
раза выше.
Мы угадали величину (она действительно связана с поломкой), но не
угадали, **как** она должна войти в модель: опасна не большая нагрузка
сама по себе, а нагрузка выше предела, и предел этот свой для каждого
класса детали.

Ручной перебор произведений упёрся в потолок. Придумывать сочетания
по одному долго, а правил, по которым ломается станок, несколько, и они
устроены по-разному: одно про мощность вне рабочего коридора, другое —
про низкий перепад **вместе с** низкими оборотами.

## Шаг 6. Все сочетания сразу

`PolynomialFeatures(degree=2, include_bias=False)` строит из переданных
столбцов все квадраты и все попарные произведения. Из пяти датчиков
получится 20 столбцов, из восьми — 44; к ним добавятся три столбца класса
детали. Модель сама разберётся, какие из них чего стоят.

Ставим его **перед** масштабированием: произведения считаем от исходных
величин, к общему масштабу приводим уже результат.

### Задание 2.6
Соберите обработку числовых столбцов из двух шагов —
`PolynomialFeatures`, затем `StandardScaler` — и обучите модель на пяти
датчиках (`ФИЗИКА`), без придуманных руками столбцов.

In [ ]:
## ── подготовка столбцов ────────────────────────────────────────────
# TODO: Pipeline из PolynomialFeatures(degree=2, include_bias=False) и StandardScaler
raise NotImplementedError("замените эту строку своим решением")

подготовка = ColumnTransformer([
    ("числа", произведения, ФИЗИКА),
    ("категории", OneHotEncoder(handle_unknown="ignore"), КАТЕГОРИИ),
])

## ── обучение ───────────────────────────────────────────────────────
model = Pipeline([("подготовка", подготовка),
                  ("модель", LogisticRegression(max_iter=3000))])
model.fit(train, train["отказ"])

## ── качество на валидации ──────────────────────────────────────────
proba = model.predict_proba(valid)[:, 1]
auc = score("6. пары пяти датчиков", valid["отказ"], proba)

## ── числа ──────────────────────────────────────────────────────────
print("Столбцов после подготовки: %d" % подготовка.transform(train.head(1)).shape[1])

## ── самопроверка ───────────────────────────────────────────────────
check(auc > 0.92, "AUC должен подскочить примерно до 0,934")

0,934 — больше, чем дали все придуманные руками столбцы вместе. Среди
двадцати произведений нашлись те, которые мы бы искали долго: например,
перепад температур вместе с оборотами (правило перегрева устроено
именно так) и квадрат момента (опасен не большой момент, а выход за
рабочий коридор в любую сторону).

### Задание 2.7
Теперь отдайте в те же произведения все восемь чисел — пять датчиков
и три посчитанных столбца — и заодно сравните качество на обучении
и на валидации.

In [ ]:
## ── подготовка столбцов ────────────────────────────────────────────
ЧИСЛА = ФИЗИКА + ["мощность", "перепад", "нагрузка"]

подготовка = ColumnTransformer([
    ("числа", произведения, ЧИСЛА),
    ("категории", OneHotEncoder(handle_unknown="ignore"), КАТЕГОРИИ),
])

## ── обучение ───────────────────────────────────────────────────────
model = Pipeline([("подготовка", подготовка),
                  ("модель", LogisticRegression(max_iter=3000))])
model.fit(train, train["отказ"])

## ── качество на валидации ──────────────────────────────────────────
proba = model.predict_proba(valid)[:, 1]
auc = score("7. пары всех восьми чисел", valid["отказ"], proba)

## ── переобучение? ──────────────────────────────────────────────────
auc_train = ...  # TODO: roc_auc_score на обучении: ответ train["отказ"] и вероятности model.predict_proba(train)[:, 1]
print("")
print("AUC на обучении:  %.3f" % auc_train)
print("AUC на валидации: %.3f" % auc)
print("Столбцов после подготовки: %d" % подготовка.transform(train.head(1)).shape[1])

## ── самопроверка ───────────────────────────────────────────────────
check(auc > 0.94, "AUC должен дойти примерно до 0,951")
check(auc_train > 0.9, "на обучении AUC должен быть около 0,957")

0,951 на валидации при 0,957 на обучении. Разрыв в шесть тысячных —
модель не переобучилась, штраф за большие веса ей не нужен.

Ручные столбцы всё-таки пригодились: с ними произведений больше
(44 против 20), и среди них нашлись полезные — например, мощность
вместе с износом. Полезны они оказались не сами по себе, а как
материал для сочетаний.

---
# Часть 3. Порог: считаем часами простоя

Модель выдаёт вероятность, а мастеру нужно решение. Решение получается
из вероятности сравнением с порогом, и порог этот берётся не из
статистики, а из цены ошибок:

* **остановили зря** — 2 часа простоя на профилактику;
* **пропустили отказ** — 16 часов простоя на аварийный ремонт.

Считать будем простой линии в часах: чем меньше, тем лучше.

### Задание 3.1
Напишите функцию, считающую часы простоя при заданном пороге, и
сравните три правила: ничего не делать, останавливать всегда и
останавливать при вероятности выше 0,5.

In [ ]:
## ── цена ошибок, часов простоя ─────────────────────────────────────
ПРОФИЛАКТИКА, АВАРИЯ = 2.0, 16.0


def простой(ответ, вероятность, порог):
    "Часы простоя линии, если останавливать станок при вероятности ≥ порога."
    стоп = вероятность >= порог
    # TODO: два слагаемых: профилактика за каждую остановку и авария за каждый пропущенный отказ
    raise NotImplementedError("замените эту строку своим решением")


## ── три правила на валидации ───────────────────────────────────────
ответ = valid["отказ"].values
print("Валидация, %d наблюдений, отказов %d:" % (len(valid), ответ.sum()))
print("   ничего не делать:      %5.0f часов" % (АВАРИЯ * ответ.sum()))
print("   останавливать всегда:  %5.0f часов" % (ПРОФИЛАКТИКА * len(valid)))
print("   останавливать при 0,5: %5.0f часов" % простой(ответ, proba, 0.5))

## ── самопроверка ───────────────────────────────────────────────────
check(простой(ответ, proba, 0.0) == ПРОФИЛАКТИКА * len(valid),
      "при нулевом пороге останавливаются все, аварий не остаётся")
check(простой(ответ, proba, 1.1) == АВАРИЯ * ответ.sum(),
      "при пороге выше единицы не останавливается никто")

Крайности плохи обе: ничего не делать — 1 088 часов простоя,
останавливать всё подряд — 4 000. Порог 0,5 даёт 732 часа, и это уже
вдвое лучше бездействия. Но 0,5 здесь ниоткуда не взялось: модель
выдаёт вероятность, а половина — это просто круглое число.

Правильный порог считается из цены ошибок. Останавливать стоит тогда,
когда ожидаемая потеря от бездействия больше, чем от остановки:

$$p \cdot 16 > 2 \qquad \Longrightarrow \qquad p > \frac{2}{16} = 0{,}125$$

### Задание 3.2
Переберите пороги от 0,01 до 0,99 с шагом 0,01, найдите тот, при
котором простой наименьший, и сравните его с расчётным.

In [ ]:
## ── перебор порогов ────────────────────────────────────────────────
сетка = np.arange(0.01, 1.0, 0.01)
часы = np.array([простой(ответ, proba, t) for t in сетка])

лучший_порог = ...  # TODO: порог из сетки, на котором часы минимальны — пригодится argmin
расчётный = ПРОФИЛАКТИКА / АВАРИЯ
ПОРОГ = ("%.2f" % лучший_порог).replace(".", ",")   # для печати по-русски

## ── числа ──────────────────────────────────────────────────────────
print("Найденный перебором порог: %s, простой %.0f часов" % (ПОРОГ, часы.min()))
print("Расчётный порог:           %.3f, простой %.0f часов"
      % (расчётный, простой(ответ, proba, расчётный)))
print("")
стоп = proba >= лучший_порог
print("При найденном пороге: остановок %d, поймано отказов %d из %d"
      % (стоп.sum(), ((ответ == 1) & стоп).sum(), ответ.sum()))

## ── график ─────────────────────────────────────────────────────────
fig, ax = plt.subplots()
ax.plot(сетка, часы, color=BLUE, lw=2)
ax.axvline(лучший_порог, color=RED, ls="--", label="найденный порог " + ПОРОГ)
ax.axhline(АВАРИЯ * ответ.sum(), color=GREY, ls=":", label="ничего не делать")
ax.set_xlabel("порог: останавливаем, если вероятность выше")
ax.set_ylabel("часы простоя за 2 000 наблюдений")
ax.set_title("Простой линии при разных порогах")
ax.legend()
plt.tight_layout()
plt.show()

## ── самопроверка ───────────────────────────────────────────────────
check(abs(лучший_порог - 0.12) < 0.02, "минимум лежит около 0,12")

0,12 против расчётных 0,125 — совпадение с точностью до шага сетки.
Это хорошая проверка на здравый смысл: формула и перебор пришли
к одному ответу, значит, мы не ошиблись ни в цене ошибок, ни в коде.

Заодно видно, во что обходится круглое число: порог 0,5 останавливал
станок 30 раз и ловил 26 отказов из 68, порог 0,12 останавливает 128 раз
и ловит 48. Остановок вчетверо больше, а простоя меньше — потому что
каждая пойманная авария экономит 16 часов, а лишняя остановка стоит 2.

## Шаг 7. Открываем тест

Тест мы не трогали. Проверим на нём и модель, и порог — оба выбраны
только по валидации.

### Задание 3.3
Посчитайте вероятности на тесте, замерьте AUC и сравните три правила
по часам простоя.

In [ ]:
## ── тест ───────────────────────────────────────────────────────────
proba_test = ...  # TODO: вероятности отказа на тесте, столбец 1
ответ_test = test["отказ"].values

score("тест: итоговая модель", ответ_test, proba_test, remember=False)

## ── часы простоя ───────────────────────────────────────────────────
print("")
print("Тест, %d наблюдений, отказов %d:" % (len(test), ответ_test.sum()))
print("   ничего не делать:        %5.0f часов" % (АВАРИЯ * ответ_test.sum()))
print("   останавливать всегда:    %5.0f часов" % (ПРОФИЛАКТИКА * len(test)))
print("   порог 0,5:               %5.0f часов" % простой(ответ_test, proba_test, 0.5))
print("   порог %s:               %5.0f часов"
      % (ПОРОГ, простой(ответ_test, proba_test, лучший_порог)))

## ── что это значит в решениях ──────────────────────────────────────
стоп_test = proba_test >= лучший_порог
print("")
print("Остановок: %d из %d, поймано отказов %d из %d"
      % (стоп_test.sum(), len(test), ((ответ_test == 1) & стоп_test).sum(),
         ответ_test.sum()))
print("Матрица ошибок:")
print(confusion_matrix(ответ_test, стоп_test))

## ── самопроверка ───────────────────────────────────────────────────
check(roc_auc_score(ответ_test, proba_test) > 0.94, "AUC на тесте около 0,960")
check(простой(ответ_test, proba_test, лучший_порог) < 600,
      "с найденным порогом простой должен быть меньше 600 часов")

482 часа против 1 088 у бездействия — простой сократился на 56 %.
Модель и порог, выбранные на валидации, перенеслись на тест без
сюрпризов: AUC 0,960 против 0,951, порог тот же.

Матрица ошибок читается так: 1 878 раз станок работал и мы его не
трогали, 54 раза остановили зря, 17 отказов пропустили, 51 поймали.
Пропущенные отказы никуда не делись — их и не должно быть ноль: ловить
все до единого означало бы останавливать станок постоянно, а это
4 000 часов простоя вместо 482.

### И последнее — про долю верных ответов

Посчитайте её для нашего решения и сравните с пустым прогнозом.

In [ ]:
## ── доля верных ответов у трёх правил ──────────────────────────────
print("Доля верных ответов на тесте:")
print("   «поломки не будет»:    %.4f" % accuracy_score(ответ_test, np.zeros(len(test))))
print("   порог 0,5:             %.4f" % accuracy_score(ответ_test, proba_test >= 0.5))
print("   наш порог %s:         %.4f"
      % (ПОРОГ, accuracy_score(ответ_test, стоп_test)))
print("")
print("Часы простоя у тех же трёх правил:")
print("   «поломки не будет»:    %5.0f" % (АВАРИЯ * ответ_test.sum()))
print("   порог 0,5:             %5.0f" % простой(ответ_test, proba_test, 0.5))
print("   наш порог %s:         %5.0f"
      % (ПОРОГ, простой(ответ_test, proba_test, лучший_порог)))

Вот ради этой таблицы стоило всё затевать. По доле верных ответов наше
решение — **худшее из трёх**: 0,9645 против 0,9660 у пустого прогноза.
По часам простоя оно лучшее: 482 против 1 088.

Никакого противоречия здесь нет. Доля верных ответов считает все ошибки
одинаковыми, а в цеху они стоят по-разному: 2 часа и 16. Модель,
которая старается угодить этой метрике, будет молчать про поломки,
потому что молчать статистически выгодно.

Метрику выбирают под решение, а не наоборот.

In [ ]:
## ── лестница ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8.6, 4.0))
names = [tag for tag, _ in LADDER]
values = [auc for _, auc in LADDER]

ax.barh(range(len(LADDER)), values, color=BLUE)
ax.set_yticks(range(len(LADDER)))
ax.set_yticklabels(names)
ax.invert_yaxis()
ax.set_xlim(0.5, 1.0)
ax.set_xlabel("ROC-AUC на валидации")
ax.set_title("Лестница улучшений")
for i, value in enumerate(values):
    ax.text(value + 0.004, i, "%.3f" % value, va="center", fontsize=10)
plt.tight_layout()
plt.show()

---
# Итоги

| Шаг | Что сделали | AUC |
|---|---|---|
| — | все столбцы подряд, включая причины отказа | 0,974 — утечка |
| 1 | пять показаний датчиков | 0,875 |
| 2 | класс детали | 0,874 — без изменений |
| 3 | мощность: момент × обороты | 0,898 |
| 4 | перепад температур | 0,898 — без изменений |
| 5 | нагрузка: износ × момент | 0,895 |
| 6 | все пары и квадраты пяти датчиков | 0,934 |
| 7 | все пары и квадраты восьми чисел | 0,951 |
| — | **тест** | **0,960** |

Решение: останавливать станок при вероятности отказа выше 0,12.
Простой на тесте — 482 часа против 1 088, если не делать ничего.

Что стоит унести из упражнения:

* **самый точный признак — самый подозрительный.** Столбцы с причинами
  отказа давали 99,85 % верных ответов и появляются только после
  вскрытия станка;
* **линейная модель умеет складывать, но не умножать.** Разность
  температур не дала ничего (модель строит её сама), произведение
  момента на обороты дало больше двух пунктов;
* **угадать величину мало — надо угадать её форму.** Нагрузка на резец
  связана с поломкой, но входит в неё через предел, свой для каждого
  класса детали, и отдельным столбцом не сработала;
* **перебор сочетаний обыгрывает интуицию.** Все пары и квадраты дали
  больше, чем три придуманных руками столбца;
* **порог берут из цены ошибок.** 2 часа против 16 дают порог 0,125,
  и перебор подтвердил: 0,12;
* **доля верных ответов — не метрика для редких событий.** Наше решение
  по ней хуже пустого прогноза, а простоя даёт вдвое меньше.